# 훈련량 예측

5장에서 저장한 평가 데이터를 바탕으로, 추천일 이전 기록을 이용해
추천일 당일부터 이후 6일까지의 7일 TSS 합계를 예측한다.

이 예측은 앞으로 기록될 훈련량을 대상으로 하며, 실제 피로·회복
상태나 개별 훈련 처방을 확정하지 않는다.


In [3]:
from pathlib import Path

import pandas as pd

project_root = Path("..").resolve()

evaluation_path = (
    project_root
    / "data"
    / "processed"
    / "goldencheetah_recommendation_evaluation.csv"
)

evaluation_df = pd.read_csv(
    evaluation_path,
    parse_dates=["recommendation_date"],
)

evaluation_df.shape

(100, 18)

In [4]:
evaluation_df = (
    evaluation_df
    .sort_values("recommendation_date")
    .reset_index(drop=True)
)

evaluation_df["recommendation_date"].agg(
    ["min", "max"]
)

min   2009-03-21 00:00:00+00:00
max   2009-06-28 00:00:00+00:00
Name: recommendation_date, dtype: datetime64[us, UTC]

## 1. 입력 후보와 정답값 선택

입력 후보는 추천일 이전 기록으로 계산한 이전 7일·28일 TSS,
이전 7일 운동 시간·라이드 횟수, 전날까지의 연속 라이드 기록일과
연속 무기록일 수로 선택한다. 무기록일은 실제 휴식일로 단정하지 않는다.

정답값 `next_7_day_tss`는 추천일 당일부터 7일간 기록된 TSS 합계다.
이후 TSS 변화량·비율은 미래 기록을 포함하므로 입력에서 제외한다.
부하 라벨과 안내 문장도 이번 입력 후보에서는 제외한다.


In [6]:
feature_columns = [
    "previous_7_day_tss",
    "previous_28_day_tss",
    "previous_7_day_workout_hours",
    "previous_7_day_ride_count",
    "previous_day_recorded_ride_streak_days",
    "previous_day_no_record_streak_days",
]

target_column = "next_7_day_tss"

prediction_df = evaluation_df[
    ["recommendation_date"]
    + feature_columns
    + [target_column]
].copy()

prediction_df.head().round(
    {
        "previous_7_day_tss": 2,
        "previous_28_day_tss": 2,
        "previous_7_day_workout_hours": 2,
        "next_7_day_tss": 2,
    }
)

,recommendation_date,previous_7_day_tss,previous_28_day_tss,previous_7_day_workout_hours,previous_7_day_ride_count,previous_day_recorded_ride_streak_days,previous_day_no_record_streak_days,next_7_day_tss
0,2009-03-21 00:00:00+00:00,829.25,2506.27,11.19,8.0,1.0,0.0,435.28
1,2009-03-22 00:00:00+00:00,926.31,2663.95,12.62,8.0,2.0,0.0,391.33
2,2009-03-23 00:00:00+00:00,812.04,2417.65,10.73,6.0,3.0,0.0,458.54
3,2009-03-24 00:00:00+00:00,709.89,2417.65,9.16,5.0,0.0,1.0,630.81
4,2009-03-25 00:00:00+00:00,580.91,2417.65,7.71,4.0,0.0,2.0,630.81
